[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P13/P13-1_Integrer_CORRIGE.ipynb)

# Intégrer et simuler

Chapitre 13 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre poutre est mesurée en onze stations, votre ressort oscille et s'amortit, votre tuyauterie à chaud relâche sa contrainte pendant des mois ? Dans chaque cas, l'effet cherché s'obtient en additionnant de petites contributions, ou en suivant un état d'un instant au suivant, et aucune formule ne le donne directement. Vous voudriez que l'ordinateur additionne et simule, et qu'il chiffre l'erreur de sa réponse ?

**Objectif terminal.** Ce chapitre montre comment, avec le module `scipy.integrate` de SciPy.

**Objectif instrumental.**

- Intégrer une courbe connue par ses points, ou une fonction, et chiffrer l'erreur.
- Résoudre une équation différentielle avec `solve_ivp` et régler sa tolérance.
- Détecter un évènement et en tirer une mesure, comme le décrément d'un oscillateur.
- Simuler un fluage et vérifier la simulation par sa solution exacte.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 à 6 et 10, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 13.1 | Simpson sur la poutre, et double intégration | 20 min |
| 13.2 | Décrément logarithmique d'un oscillateur | 20 min |
| 13.3 | Contrainte de Norton à mille heures (défi) | 20 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scipy", version("scipy"))

Python 3.12.0
numpy 2.4.6
scipy 1.18.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [3]:
import math
import numpy as np
from scipy.integrate import cumulative_trapezoid, quad, simpson, solve_ivp, trapezoid

### Section 13.1 : intégrer une courbe mesurée.

In [4]:
# notion: quadrature
x = np.linspace(0.0, np.pi, 11)
valeurs = np.sin(x)                                         # une courbe connue par 11 points
aire_trapeze = trapezoid(valeurs, x)                        # somme de trapèzes, ordre 2
aire_simpson = simpson(valeurs, x=x)                        # paraboles par trois points, ordre 4
aire_quad, erreur_estimee = quad(np.sin, 0.0, np.pi)        # quad rend la valeur et une estimation de son erreur
cumulee = cumulative_trapezoid(valeurs, x, initial=0.0)     # l'aire jusqu'à chaque point, même longueur que x
print(f"aire sous sin de 0 à pi : trapèzes {aire_trapeze:.5f} ; Simpson {aire_simpson:.6f} ; quad {aire_quad:.6f} ; exacte 2")
assert abs(aire_trapeze - 2.0) < 0.02 and abs(aire_simpson - 2.0) < 2e-4, "ordre 2 contre ordre 4"
assert abs(aire_quad - 2.0) < 1e-12 and erreur_estimee < 1e-12, "quad : de l'ordre de l'arrondi"
assert len(cumulee) == len(x) and cumulee[0] == 0.0 and abs(cumulee[-1] - aire_trapeze) < 1e-12
erreurs_pas = []

for n_points in (11, 21, 41):
    xn = np.linspace(0.0, np.pi, n_points)
    erreurs_pas.append(abs(trapezoid(np.sin(xn), xn) - 2.0))

print("erreur des trapèzes pour 11, 21 et 41 points :", [round(float(e), 5) for e in erreurs_pas])

aire sous sin de 0 à pi : trapèzes 1.98352 ; Simpson 2.000110 ; quad 2.000000 ; exacte 2
erreur des trapèzes pour 11, 21 et 41 points : [0.01648, 0.00411, 0.00103]


In [5]:
assert (
    abs(erreurs_pas[0] / erreurs_pas[1] - 4.0) < 0.1
), "pas divisé par deux, erreur divisée par quatre"

## Exercice 13.1 : Simpson sur la poutre, et double intégration (20 min)

Reprenez la flèche de la poutre. Calculez l'écart de `simpson` avec les onze stations : est-il meilleur que celui des trapèzes ? Refaites le calcul avec vingt et une stations. Retrouvez ensuite la flèche par double intégration de la courbure $M/(E\,I)$ avec `cumulative_trapezoid`, sur $401$ stations : intégrez la courbure en pente, la pente en flèche, puis imposez une flèche nulle à l'appui de droite.

**Résultat attendu.** Avec 11 stations, Simpson s'écarte de 0,0744 mm, plus que les trapèzes (0,0372 mm), parce que le moment change de pente au milieu ; avec vingt et une stations, Simpson est exact à l'arrondi ; la double intégration sur 401 stations donne 1,8601 mm.

**Seconde voie.** La formule fermée $F L^3/(48\,E I)$ du chapitre 3 donne 1,8601 mm.

**Indication.** Simpson ajuste des paraboles sur des paires d'intervalles. Avec onze stations, une parabole enjambe le milieu, où la courbure change brutalement : elle s'y trompe. Avec vingt et une stations, le milieu tombe entre deux paraboles et l'intégrale est exacte. Pour la double intégration, `pente = cumulative_trapezoid(courbure, x, initial=0.0)`, puis la même fonction sur `pente` ; la flèche libre ne vaut pas zéro à droite, on retire la droite qui l'annule.

Les contrôles de la cellule lisent les variables `ecart_quad`, `ecart_simpson`, `ecart_simpson_21`, `ecart_trapeze`, `fleche_exacte_mm`, `y_milieu` : gardez ces noms.

In [6]:
F_N, L_mm, E_MPa, I_mm4 = 2000.0, 1000.0, 210000.0, 20.0 * 40.0**3 / 12
# Solution :
def integrande(xs):
    """F m(x)^2 / (E I), en mm/mm : la contribution de chaque tranche de la poutre à la flèche."""
    m = np.minimum(xs, L_mm - xs) / 2
    return F_N * m**2 / (E_MPa * I_mm4)

fleche_exacte_mm = F_N * L_mm**3 / (48 * E_MPa * I_mm4)
stations = np.linspace(0.0, L_mm, 11)                       # 11 stations, une tous les 100 mm
mesures = integrande(stations)
fleche_trapeze = trapezoid(mesures, stations)
fleche_simpson = simpson(mesures, x=stations)
fleche_quad = quad(integrande, 0.0, L_mm)[0]
ecart_trapeze = abs(fleche_trapeze - fleche_exacte_mm)
ecart_simpson = abs(fleche_simpson - fleche_exacte_mm)
ecart_quad = abs(fleche_quad - fleche_exacte_mm)
stations_21 = np.linspace(0.0, L_mm, 21)
ecart_simpson_21 = abs(simpson(integrande(stations_21), x=stations_21) - fleche_exacte_mm)
print(f"flèche exacte : {fleche_exacte_mm:.4f} mm")
print(f"écarts avec 11 stations : trapèzes {ecart_trapeze:.4f} mm ; Simpson {ecart_simpson:.4f} mm ; quad sous 1e-12 :", ecart_quad < 1e-12)
print("Simpson avec 21 stations, écart sous 1e-12 mm :", ecart_simpson_21 < 1e-12)
# Fin de la solution
assert abs(fleche_exacte_mm - 1.8601) < 5e-5, "flèche exacte attendue : 1,8601 mm"
assert abs(ecart_trapeze - 0.0372) < 5e-5, "écart des trapèzes attendu : 0,0372 mm"
assert abs(ecart_simpson - 0.0744) < 5e-5, "écart de Simpson attendu : 0,0744 mm"
assert ecart_quad < 1e-12, "quad retrouve la valeur exacte à l'arrondi près"
assert (
    ecart_simpson_21 < 1e-12
), "avec 21 stations, la cassure tombe entre deux paraboles : Simpson est exact"
assert ecart_simpson > ecart_trapeze, "avec 11 stations, la cassure du moment gêne Simpson"
# Solution :
# La même flèche par double intégration de la courbure : la pente, puis la flèche, par cumulative_trapezoid.
x_400 = np.linspace(0.0, L_mm, 401)
courbure = F_N * np.minimum(x_400, L_mm - x_400) / 2 / (E_MPa * I_mm4)       # M / (E I), en 1/mm
pente = cumulative_trapezoid(courbure, x_400, initial=0.0)
fleche_libre = cumulative_trapezoid(pente, x_400, initial=0.0)              # sans la condition d'appui à droite
pente_depart = -fleche_libre[-1] / L_mm                                       # impose y(L) = 0
y_milieu = fleche_libre[200] + pente_depart * x_400[200]
print(f"flèche par double intégration (401 stations) : {-y_milieu:.4f} mm")
# Fin de la solution
assert abs(-y_milieu - 1.8601) < 5e-5, "double intégration attendue : 1,8601 mm"
print("Exercice 13.1 : contrôles passés")

flèche exacte : 1.8601 mm
écarts avec 11 stations : trapèzes 0.0372 mm ; Simpson 0.0744 mm ; quad sous 1e-12 : True
Simpson avec 21 stations, écart sous 1e-12 mm : True
flèche par double intégration (401 stations) : 1.8601 mm
Exercice 13.1 : contrôles passés


### Section 13.2 : une équation différentielle se résout par petits pas.

In [7]:
# notion: solve-ivp
# Oscillateur : m x'' + c x' + k x = 0, lâché sans vitesse d'une élongation de 1 mm (x en mm, v en mm/s, t en s).
m_kg, k_Nm, zeta = 5.0, 20000.0, 0.05
c_Nsm = 2 * zeta * math.sqrt(k_Nm * m_kg)
x0_mm, v0_mms = 1.0, 0.0

def derivee(t, etat):
    """Dérivée de l'état (x, v) : x' = v et v' = -(c v + k x) / m."""
    pos, vit = etat
    return [vit, -(c_Nsm * vit + k_Nm * pos) / m_kg]

def euler(h, t_fin):
    """Méthode d'Euler : à chaque pas, x et v avancent le long de leurs pentes. Rend la position à t_fin."""
    pos, vit = x0_mm, v0_mms
    for _ in range(int(round(t_fin / h))):
        pente_x, pente_v = derivee(0.0, [pos, vit])
        pos, vit = pos + h * pente_x, vit + h * pente_v
    return pos

In [8]:
omega0 = math.sqrt(k_Nm / m_kg)
omega_d = omega0 * math.sqrt(1 - zeta**2)

def exacte(t):
    """Solution exacte du sous-amorti, lâché sans vitesse : enveloppe exponentielle fois cosinus et sinus."""
    t = np.asarray(t, dtype=float)
    return x0_mm * np.exp(-zeta * omega0 * t) * (np.cos(omega_d * t) + zeta * omega0 / omega_d * np.sin(omega_d * t))

erreur_euler = abs(euler(0.005, 0.05) - exacte(0.05))
erreur_euler_demi = abs(euler(0.0025, 0.05) - exacte(0.05))
rapport_euler = erreur_euler / erreur_euler_demi
print(f"Euler à 50 ms : écart {erreur_euler:.4f} mm avec h = 5 ms, {erreur_euler_demi:.4f} mm avec h = 2,5 ms ; rapport {rapport_euler:.2f}")
assert (
    abs(erreur_euler - 0.5341) < 5e-5 and abs(erreur_euler_demi - 0.2421) < 5e-5
), "écarts d'Euler"
assert (
    2.0 < rapport_euler < 2.4
), "ordre 1 : pas divisé par deux, écart divisé par un peu plus de deux"

Euler à 50 ms : écart 0.5341 mm avec h = 5 ms, 0.2421 mm avec h = 2,5 ms ; rapport 2.21


In [9]:
solution = solve_ivp(derivee, (0.0, 0.5), [x0_mm, v0_mms], rtol=1e-9, atol=1e-12, dense_output=True)
t_grille = np.linspace(0.0, 0.5, 601)
ecart_integrateur = float(np.max(np.abs(solution.sol(t_grille)[0] - exacte(t_grille))))
print("solve_ivp : état final", np.round(solution.y[:, -1], 4), "; écart maximal sous 1e-8 mm :", ecart_integrateur < 1e-8)
assert solution.success and solution.y.shape[0] == 2, "deux variables d'état"
assert (
    8.5e-10 < ecart_integrateur < 9.6e-10
), "écart à la solution exacte attendu : 9,1 x 10^-10 mm"
pas_selon_tolerance = {}

for tolerance in (1e-3, 1e-6, 1e-9):
    essai = solve_ivp(derivee, (0.0, 0.5), [x0_mm, v0_mms], rtol=tolerance, atol=1e-12)
    pas_selon_tolerance[tolerance] = len(essai.t) - 1

print("pas acceptés selon la tolérance relative :", pas_selon_tolerance)
assert (
    list(pas_selon_tolerance.values()) == sorted(pas_selon_tolerance.values())
), "tolérance fine, plus de pas"

solve_ivp : état final [ 0.2046 -2.1695] ; écart maximal sous 1e-8 mm : True
pas acceptés selon la tolérance relative : {0.001: 45, 1e-06: 162, 1e-09: 637}


In [10]:
# notion: oscillateur
# Les maxima successifs de x(t) sont les passages de la vitesse de plus à moins : un évènement de solve_ivp.
def maximum(t, etat):
    return etat[1]

maximum.direction = -1                                        # on ne retient que les passages de + vers -
pics = solve_ivp(derivee, (0.0, 0.6), [x0_mm, v0_mms], rtol=1e-11, atol=1e-13, events=maximum)
x_pics = pics.y_events[0][:, 0]
t_pics = pics.t_events[0]
decrement = math.log(x_pics[0] / x_pics[1])
decrement_theorique = 2 * math.pi * zeta / math.sqrt(1 - zeta**2)
print(f"maxima : {len(x_pics)} ; décrément logarithmique : {decrement:.4f} ; théorique : {decrement_theorique:.4f}")
assert len(x_pics) >= 3 and abs(x_pics[0] - x0_mm) < 1e-9, "le premier maximum est le départ"
assert abs(decrement - decrement_theorique) < 1e-8, "le décrément mesuré redonne la formule"

maxima : 7 ; décrément logarithmique : 0.3146 ; théorique : 0.3146


## Exercice 13.2 : Décrément logarithmique d'un oscillateur (20 min)

Pour l'oscillateur du texte, définissez un évènement dont la fonction est la vitesse, avec `direction = -1`, et intégrez de $0$ à $0{,}6$ s avec une tolérance fine. Lisez la position aux deux premiers maxima et calculez le décrément logarithmique $\delta$. Calculez aussi la fréquence propre $f_0$ et la pseudo-période.

**Résultat attendu.** Décrément de 0,3146 ; fréquence propre de 10,07 Hz ; pseudo-période de 99,5 ms, soit une pseudo-fréquence de 10,05 Hz.

**Seconde voie.** La formule fermée $2\pi\,\zeta/\sqrt{1-\zeta^2}$ donne 0,3146.

**Indication.** La fonction d'évènement renvoie `etat[1]`, la vitesse ; avec `direction = -1`, on ne retient que les passages de positif à négatif, c'est-à-dire les maxima de $x$. Les positions sont dans `y_events[0][:, 0]` et les instants dans `t_events[0]`. Le décrément est `math.log(x[0] / x[1])`, la pseudo-période `t[1] - t[0]`, et la fréquence propre $\sqrt{k/m}/(2\pi)$.

Les contrôles de la cellule lisent les variables `f0_Hz`, `fd_Hz` : gardez ces noms.

In [11]:
# Solution :
f0_Hz = math.sqrt(k_Nm / m_kg) / (2 * math.pi)
pseudo_periode_s = float(t_pics[1] - t_pics[0])
fd_Hz = 1.0 / pseudo_periode_s
print(f"décrément {decrement:.4f} ; fréquence propre {f0_Hz:.2f} Hz ; pseudo-période {1e3 * pseudo_periode_s:.2f} ms ({fd_Hz:.2f} Hz)")
# Fin de la solution
assert abs(decrement - 0.3146) < 5e-5, "décrément attendu : 0,3146"
assert abs(f0_Hz - 10.07) < 0.005, "fréquence propre attendue : 10,07 Hz"
assert (
    abs(fd_Hz - f0_Hz * math.sqrt(1 - zeta**2)) < 1e-6
), "pseudo-fréquence : f0 racine(1 - zeta^2)"
print("Exercice 13.2 : contrôles passés")

décrément 0.3146 ; fréquence propre 10.07 Hz ; pseudo-période 99.47 ms (10.05 Hz)
Exercice 13.2 : contrôles passés


### Section 13.3 : le fluage de Norton se simule à pas adaptatif.

In [12]:
# notion: norton
# Longueur totale bloquée : l'allongement de fluage, qui croît comme A sigma^n, relâche la contrainte,
# donc sigma' = -E A sigma^n. Acier à chaud : E = 200 000 MPa, A = 1e-20 MPa^-5 par heure, n = 5, sigma(0) = 300 MPa.
E_fluage_MPa, A_norton, n_norton, sigma0_MPa = 200000.0, 1e-20, 5, 300.0

def relaxation(t, sigma):
    """sigma' = -E A sigma^n : la contrainte diminue d'autant plus vite qu'elle est grande."""
    return [-E_fluage_MPa * A_norton * sigma[0]**n_norton]

def sigma_exacte(t):
    """Solution par séparation des variables : (sigma0^(1-n) + (n-1) E A t)^(1/(1-n))."""
    return (sigma0_MPa**(1 - n_norton) + (n_norton - 1) * E_fluage_MPa * A_norton * t)**(1.0 / (1 - n_norton))

relache = solve_ivp(relaxation, (0.0, 1000.0), [sigma0_MPa], rtol=1e-11, atol=1e-13, t_eval=[100.0, 500.0, 1000.0])
print("contrainte à 100, 500 et 1 000 h (MPa) :", np.round(relache.y[0], 2))
assert (
    np.allclose(relache.y[0], sigma_exacte(np.array([100.0, 500.0, 1000.0])), rtol=0, atol=1e-6)
), "solution exacte"
assert abs(relache.y[0][-1] - 295.33) < 0.005, "contrainte à 1 000 h attendue : 295,33 MPa"

contrainte à 100, 500 et 1 000 h (MPa) : [299.52 297.62 295.33]


## Exercice 13.3 : Contrainte de Norton à mille heures (défi, 20 min)

Reprenez la relaxation de Norton ($E = 200\,000$ MPa, $A = 10^{-20}$ MPa$^{-5}$ par heure, $n = 5$, $\sigma_0 = 300$ MPa). Calculez la contrainte à mille heures. Comptez ensuite les pas acceptés pour des tolérances relatives de $10^{-3}$, $10^{-6}$ et $10^{-10}$, avec `atol=1e-13`, et vérifiez que l'écart à la solution exacte reste minuscule, même à tolérance large.

**Résultat attendu.** Contrainte de 295,33 MPa à mille heures ; 4 pas acceptés à $10^{-3}$ et 6 à $10^{-10}$ ; écart à la solution exacte inférieur à $10^{-8}$ MPa dans les trois cas.

**Seconde voie.** La séparation des variables dans l'équation de Norton donne 295,33 MPa à mille heures.

**Indication.** Ajoutez `1000.0` à `t_eval`, puis lisez `relache.y[0, -1]`. Une boucle sur les trois tolérances, sans `t_eval`, garde dans `essai.t` les pas acceptés : elle les compte, `len(essai.t)` moins un, et compare `essai.y[0, -1]` à `sigma_exacte(1000.0)`. Quatre à six pas suffisent : la solution est lisse, et le pas adaptatif ne s'inquiète pas des tolérances larges.

Les contrôles de la cellule lisent la variable `sigma_1000` : gardez ce nom.

In [13]:
pas_norton = {}
# Solution :
for tolerance in (1e-3, 1e-6, 1e-10):
    essai = solve_ivp(relaxation, (0.0, 1000.0), [sigma0_MPa], rtol=tolerance, atol=1e-13)
    pas_norton[tolerance] = len(essai.t) - 1
    assert (
        abs(essai.y[0, -1] - sigma_exacte(1000.0)) < 1e-8
    ), "même à tolérance large, l'écart reste minuscule"

sigma_1000 = float(relache.y[0][-1])
print(f"contrainte à 1 000 h : {sigma_1000:.2f} MPa ; pas adaptatifs : {pas_norton}")
# Fin de la solution
assert abs(sigma_1000 - 295.33) < 0.005, "contrainte de Norton attendue : 295,33 MPa"
assert all(n < 10 for n in pas_norton.values()), "quelques pas suffisent : la solution est lisse"
print("Exercice 13.3 : contrôles passés")

contrainte à 1 000 h : 295.33 MPa ; pas adaptatifs : {0.001: 4, 1e-06: 5, 1e-10: 6}
Exercice 13.3 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F13-1 : Intégrale définie (`scipy.integrate.quad`)

In [14]:
import numpy as np
from scipy.integrate import quad

k, b = 50e3, 20e6                       # raideurs, N/m et N/m3
x_max = 0.05                            # écrasement maximal, m

def force(x):
    return k * x + b * x**3             # effort de compression, N

travail, erreur = quad(force, 0.0, x_max)
exact = k * x_max**2 / 2 + b * x_max**4 / 4
print(f"Travail absorbé : {travail:.2f} J".replace(".", ","))
print("Contrôle analytique :", np.isclose(travail, exact))

Travail absorbé : 93,75 J
Contrôle analytique : True


### Fiche F13-2 : Équations différentielles (`scipy.integrate.solve_ivp`)

In [15]:
import numpy as np
from scipy.integrate import solve_ivp

def derivee(t, etat):                     # g = 9,81 m/s² et L = 0,5 m
    return [etat[1], -(9.81 / 0.5) * np.sin(etat[0])]

sol = solve_ivp(derivee, (0.0, 3.0), [2.0, 0.0], rtol=1e-8, atol=1e-10)
en = 0.5 * (0.5 * sol.y[1])**2 + 9.81 * 0.5 * (1 - np.cos(sol.y[0]))
variation = np.max(np.abs(en - en[0]))
print(f"pas acceptés : {len(sol.t) - 1}")
print(f"variation d'énergie : {variation:.2e} J/kg".replace(".", ","))
print(np.isclose(variation, 0.0, atol=1e-6))

pas acceptés : 132
variation d'énergie : 1,24e-07 J/kg
True
